# 04 — RAG over ASIC RG 271

## Purpose

This notebook builds the first retrieval layer for the complaint intelligence platform.

The goal is to use ASIC Regulatory Guide 271 as a small knowledge base for internal dispute resolution guidance.

The workflow is:
- Load the ASIC RG 271 PDF
- Extract text from the PDF
- Split the text into smaller chunks
- Convert chunks into embeddings
- Store embeddings in a local Chroma vector database
- Retrieve the most relevant RG 271 chunks for a redacted complaint narrative

This creates the foundation for a RAG assistant that can support complaint triage with regulatory context.

In [1]:
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 300)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PDF_PATH = Path("../data/knowledge_base/rg271.pdf")

print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH)

PDF exists: True
PDF path: ../data/knowledge_base/rg271.pdf


In [3]:
reader = PdfReader(PDF_PATH)

num_pages = len(reader.pages)

print("Number of pages:", num_pages)

Number of pages: 57


In [4]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    
    if text:
        pages.append({
            "page_number": page_number,
            "text": text
        })

pages_df = pd.DataFrame(pages)

print("Pages with extracted text:", len(pages_df))

pages_df.head()

Pages with extracted text: 57


,page_number,text
0,1,"\n \n \nREGULATORY GUIDE 271 \nInternal dispute resolution \nSeptember 2021 \nAbout this guide \nThis guide is for Australian financial services (AFS) licensees, unlicensed \nproduct issuers, unlicensed secondary sellers, trustees of regulated \nsuperannuation funds (other than self-managed sup..."
1,2,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 2 \nAbout ASIC regulatory documents \nIn administering legislation ASIC issues the following types of regulatory \ndocuments. \nConsultation papers: seek feedback from stak...
2,3,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 3 \nContents \nA Overview ................................................................................................. 4 \nFinancial services dispute resolution framew...
3,4,REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 4 \nA Overview \nKey points \nFinancial firms must have a dispute resolution system that consists of: \n• an internal dispute resolution (IDR) procedure that meets the sta...
4,5,"REGULATORY GUIDE 271: Internal dispute resolution \n© Australian Securities and Investments Commission September 2021 Page 5 \nNote 2: Unlicensed carried over instrument lenders (unlicensed COI lenders) have IDR \nobligations, but are not required to be a member of AFCA (see RG 271.3). \nRG 271..."


In [5]:
print(pages_df.loc[0, "text"][:1500])

 
 
 
REGULATORY GUIDE 271 
Internal dispute resolution 
September 2021 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders).  
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We will withdraw 
RG 165 on 5 October 2022. 
T

In [6]:
full_text = "\n\n".join(pages_df["text"].tolist())

print("Total characters:", len(full_text))
print("Total words:", len(full_text.split()))
print(full_text[:1000])

Total characters: 126909
Total words: 18635
 
 
 
REGULATORY GUIDE 271 
Internal dispute resolution 
September 2021 
About this guide 
This guide is for Australian financial services (AFS) licensees, unlicensed 
product issuers, unlicensed secondary sellers, trustees of regulated 
superannuation funds (other than self-managed superannuation funds 
(SMSFs)), trustees of approved deposit funds, retirement savings account 
providers, Australian credit licensees (credit licensees) and unlicensed 
carried over instrument lenders (unlicensed COI lenders).  
The standards and requirements highlighted in this guide are enforceable. 
It explains what these financial firms must do to have an internal dispute 
resolution (IDR) system in place that meets ASIC’s standards and 
requirements. 
Note: This guide comes into effect on 5 October 2021. For complaints 
received by financial firms before that date, Regulatory Guide 165 Licensing: 
Internal and external dispute resolution (RG 165) applies. We

In [7]:
import re

def clean_pdf_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    # Birden çok boşluk, yeni satır veya sekme karakterini tek bir boşluğa dönüştürür.
    text = text.replace(" .", ".")
    # Noktadan önceki gereksiz boşluğu kaldırır.
    text = text.replace(" ,", ",")
    # Virgülden önceki gereksiz boşluğu kaldırır.
    return text.strip()
    # Metnin başındaki ve sonundaki boşlukları temizler.

clean_text = clean_pdf_text(full_text)

print("Clean characters:", len(clean_text))
print(clean_text[:1000])

clean_text = clean_pdf_text(full_text)

print("Clean characters:", len(clean_text))
print(clean_text[:1000])

Clean characters: 124357
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 October 2022. T

In [ ]:
def recursive_chunk_text(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 150,
    separators: list[str] = ["\n\n", "\n", ". ", "; ", ", ", " "]
) -> list[str]:
    chunks = []
    # sonuçları saklamak için boş bir liste oluşturulur
    
    def split_recursive(segment: str, seps: list[str]):
        # bir segmenti ayırmak için iç fonksiyon
        if len(segment) <= chunk_size:
            # segment boyutu sınıra uygunsa olduğu gibi döndür
            return [segment.strip()]
        
        if not seps:
            # ayırıcı kalmadıysa düz olarak parça parça ayır
            return [
                segment[i:i + chunk_size].strip()
                for i in range(0, len(segment), chunk_size - chunk_overlap)
            ]
        
        sep = seps[0]
        # ilk ayırıcıyı seç
        parts = segment.split(sep)
        # segmenti seçilen ayırıcıyla böl
        
        temp_chunks = []
        current = ""
        # geçici parça listesi ve birleştirme değişkeni
        
        for part in parts:
            candidate = current + sep + part if current else part
            # mevcut parça ile yeni parçayı birleştir
            
            if len(candidate) <= chunk_size:
                current = candidate
                # eğer birleştirilmiş parça sınıra sığarsa devam et
            else:
                if current:
                    temp_chunks.append(current.strip())
                    # mevcut parçayı kaydet
                current = part
                # yeni parçayı başlat
        
        if current:
            temp_chunks.append(current.strip())
            # döngü sonunda kalan parçayı ekle
        
        final = []
        for chunk in temp_chunks:
            if len(chunk) > chunk_size:
                final.extend(split_recursive(chunk, seps[1:]))
                # parça hala çok büyükse sonraki ayırıcılarla tekrar ayır
            else:
                final.append(chunk)
                # uygun boyuttaysa final listesine ekle
        
        return final
    
    base_chunks = split_recursive(text, separators)
    # metni başlangıç ayırıcılara göre parçala
    
    for i, chunk in enumerate(base_chunks):
        if i == 0:
            chunks.append(chunk)
            # ilk parçayı doğrudan ekle
        else:
            previous_tail = chunks[-1][-chunk_overlap:]
            # önceki parçanın son bölümünü al
            chunks.append((previous_tail + " " + chunk).strip())
            # önceki parça ile örtüşmeyi ekle
    
    return chunks
    # sonuç listesini döndür

recursive_chunks = recursive_chunk_text(clean_text)

print("Recursive chunks:", len(recursive_chunks))
print("First chunk length:", len(recursive_chunks[0]))
print(recursive_chunks[0][:1200])


recursive_chunks = recursive_chunk_text(clean_text)

print("Recursive chunks:", len(recursive_chunks))
print("First chunk length:", len(recursive_chunks[0]))
print(recursive_chunks[0][:1200])

Recursive chunks: 125
First chunk length: 1108
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 16

In [9]:
for i, chunk in enumerate(recursive_chunks[:5]):
    print(f"\n--- Recursive Chunk {i} ---")
    print("Length:", len(chunk))
    print(chunk[:1000])


--- Recursive Chunk 0 ---
Length: 1108
REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders). The standards and requirements highlighted in this guide are enforceable. It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements. Note: This guide comes into effect on 5 October 2021. For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies. We will withdraw RG 165 on 5 

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [11]:
def split_into_sentences(text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]
    return sentences

sentences = split_into_sentences(clean_text)

print("Number of sentences:", len(sentences))
print(sentences[:5])

Number of sentences: 524
['REGULATORY GUIDE 271 Internal dispute resolution September 2021 About this guide This guide is for Australian financial services (AFS) licensees, unlicensed product issuers, unlicensed secondary sellers, trustees of regulated superannuation funds (other than self-managed superannuation funds (SMSFs)), trustees of approved deposit funds, retirement savings account providers, Australian credit licensees (credit licensees) and unlicensed carried over instrument lenders (unlicensed COI lenders).', 'The standards and requirements highlighted in this guide are enforceable.', 'It explains what these financial firms must do to have an internal dispute resolution (IDR) system in place that meets ASIC’s standards and requirements.', 'Note: This guide comes into effect on 5 October 2021.', 'For complaints received by financial firms before that date, Regulatory Guide 165 Licensing: Internal and external dispute resolution (RG 165) applies.']


In [13]:
sentence_embeddings = embedding_model.encode(
    sentences,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Sentence embeddings shape:", sentence_embeddings.shape)


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Sentence embeddings shape: (524, 384)
